In [1]:
import numpy as np
from tqdm import tqdm_notebook as tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
# Load the data
from sklearn.datasets import load_breast_cancer
import pandas as pd
from tqdm import tqdm_notebook as tqdm

In [2]:
!wget https://raw.githubusercontent.com/datasets/breast-cancer/master/data/breast-cancer.csv

--2025-04-02 19:13:13--  https://raw.githubusercontent.com/datasets/breast-cancer/master/data/breast-cancer.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20203 (20K) [text/plain]
Saving to: ‘breast-cancer.csv.1’

breast-cancer.csv.1 100%[===================>]  19,73K  --.-KB/s    in 0s      

2025-04-02 19:13:14 (82,7 MB/s) - ‘breast-cancer.csv.1’ saved [20203/20203]



In [3]:
!jupyter nbextension enable --py widgetsnbextension

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

optional arguments:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: console dejavu events execute kernel kernelspec lab
labextension labhub migrate nbconvert run server troubleshoot trust

Jupyter command `jupyter-nbextension` not found.


In [4]:
data = load_breast_cancer()


X = data.data[:, :-1]
y = data.data[:, -1]
y = y.reshape((-1, 1))


In [5]:

# Preprocess the data
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [6]:

class NeuralNetwork:
    
    def __init__(self, num_features, hidden_units):
        self.num_features = num_features
        self.hidden_units = hidden_units
        self.weights1 = np.random.randn(num_features, hidden_units)
        self.bias1 = np.zeros((1, hidden_units))
        self.weights2 = np.random.randn(hidden_units, 1)
        self.bias2 = np.zeros((1, 1))
        
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def forward(self, X):
        z1 = np.dot(X, self.weights1) + self.bias1
        a1 = self.sigmoid(z1)
        z2 = np.dot(a1, self.weights2) + self.bias2
        y_pred = self.sigmoid(z2)
        return y_pred, a1
    
    def compute_cost(self, y_pred, y_true):
        m = y_true.shape[0]
        cost = -(1/m) * np.sum(y_true*np.log(y_pred) + (1-y_true)*np.log(1-y_pred))
        return cost
    
    def backward(self, X, y_pred, y_true, learning_rate, a1):
        m = y_true.shape[0]
        dz2 = y_pred - y_true
        dw2 = (1/m) * np.dot(a1.T, dz2)
        db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True)
        dz1 = np.dot(dz2, self.weights2.T) * a1 * (1 - a1)
        dw1 = (1/m) * np.dot(X.T, dz1)
        db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True)
        self.weights2 = self.weights2 - learning_rate * dw2
        self.bias2 = self.bias2 - learning_rate * db2
        self.weights1 = self.weights1 - learning_rate * dw1
        self.bias1 = self.bias1 - learning_rate * db1
    
    def sigmoid_prime(self, z):
        return self.sigmoid(z) * (1 - self.sigmoid(z))
        
    def train(self, X, y_true, learning_rate, num_iterations):
        for i in tqdm(range(num_iterations)):
            y_pred, a1 = self.forward(X)
            cost = self.compute_cost(y_pred, y_true)
            self.backward(X, y_pred, y_true, learning_rate, a1)
            if i % 1000 == 0:
                print(f"Cost after iteration {i}: {cost:.4f}")

In [8]:
nn = NeuralNetwork(X_train.shape[1], 10)
print(X_train.shape)
print(y_train.shape)
nn.train(X_train, y_train, 0.1, 10000)

(455, 29)
(455, 1)


/var/folders/49/7n60k7011n11zcdptxxpmvvh0000gn/T/ipykernel_42407/2492870946.py:43: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i in tqdm(range(num_iterations)):


  0%|          | 0/10000 [00:00<?, ?it/s]

Cost after iteration 0: 1.8119
Cost after iteration 1000: 0.2923
Cost after iteration 2000: 0.2897
Cost after iteration 3000: 0.2885
Cost after iteration 4000: 0.2879
Cost after iteration 5000: 0.2875
Cost after iteration 6000: 0.2873
Cost after iteration 7000: 0.2871
Cost after iteration 8000: 0.2870
Cost after iteration 9000: 0.2869


In [9]:
# Evaluate the neural network on the test set
y_pred = nn.forward(X_test)[0]

y_pred_e = np.round(y_pred,2)
y_test_e = np.round(y_test,2)
print (y_pred_e)
print(y_test_e)
accuracy = np.mean(y_pred_e == y_test_e)
print(f"Test accuracy: {accuracy}")

[[0.09]
 [0.07]
 [0.08]
 [0.09]
 [0.09]
 [0.12]
 [0.09]
 [0.09]
 [0.12]
 [0.09]
 [0.07]
 [0.06]
 [0.07]
 [0.09]
 [0.09]
 [0.08]
 [0.07]
 [0.06]
 [0.11]
 [0.09]
 [0.1 ]
 [0.07]
 [0.12]
 [0.08]
 [0.06]
 [0.09]
 [0.08]
 [0.07]
 [0.07]
 [0.11]
 [0.09]
 [0.08]
 [0.1 ]
 [0.08]
 [0.09]
 [0.07]
 [0.08]
 [0.07]
 [0.08]
 [0.07]
 [0.08]
 [0.07]
 [0.08]
 [0.08]
 [0.06]
 [0.09]
 [0.08]
 [0.08]
 [0.1 ]
 [0.06]
 [0.07]
 [0.1 ]
 [0.1 ]
 [0.1 ]
 [0.1 ]
 [0.07]
 [0.1 ]
 [0.09]
 [0.09]
 [0.08]
 [0.08]
 [0.1 ]
 [0.07]
 [0.07]
 [0.07]
 [0.1 ]
 [0.11]
 [0.11]
 [0.06]
 [0.09]
 [0.08]
 [0.11]
 [0.06]
 [0.1 ]
 [0.11]
 [0.09]
 [0.07]
 [0.08]
 [0.07]
 [0.09]
 [0.09]
 [0.08]
 [0.07]
 [0.09]
 [0.07]
 [0.13]
 [0.13]
 [0.11]
 [0.09]
 [0.09]
 [0.09]
 [0.09]
 [0.09]
 [0.05]
 [0.06]
 [0.08]
 [0.08]
 [0.12]
 [0.07]
 [0.07]
 [0.07]
 [0.09]
 [0.1 ]
 [0.09]
 [0.1 ]
 [0.07]
 [0.1 ]
 [0.07]
 [0.08]
 [0.07]
 [0.06]
 [0.08]
 [0.08]
 [0.07]]
[[0.09]
 [0.07]
 [0.08]
 [0.09]
 [0.08]
 [0.12]
 [0.09]
 [0.08]
 [0.1 ]
 [0.08]
 [0.07]